In [3]:
import pandas as pd
import re

# 1️⃣ Load dataset
df = pd.read_csv("/content/sample_data/IMDB Dataset.csv")
print("Columns:", df.columns.tolist())
print(df.head())

# 2️⃣ Define stopwords
stopwords = set([
    'i','me','my','we','our','you','he','she','it','they',
    'the','a','an','and','but','or','if','to','of','in','on','for','with','as',
    'this','that','is','was','are','be','have','has','had','at','by','from',
    'so','not','can','will','all','one','like','film','movie','about','your',
    'myself','him','her','them','their','its','our','ours','yourself'
])

# 3️⃣ Split positive & negative reviews
positive_reviews = df[df['sentiment'] == 'positive'].copy()
negative_reviews = df[df['sentiment'] == 'negative'].copy()

# 4️⃣ Clean and tokenize review
def clean_review_words(text):
    text = text.lower()
    text = re.sub(r'[^a-z\s]', ' ', text)  # remove punctuation
    words = text.split()
    words = [w for w in words if w not in stopwords and len(w) > 2]
    return words

# 5️⃣ Pre-clean reviews to speed up counting
positive_reviews['clean_words'] = positive_reviews['review'].apply(clean_review_words)
negative_reviews['clean_words'] = negative_reviews['review'].apply(clean_review_words)

# 6️⃣ Count number of reviews containing each keyword (max 1 per review)
def keyword_review_counts(reviews, keywords):
    counts = {}
    for k in keywords:
        counts[k] = int(reviews['clean_words'].apply(lambda words: k in words).sum())
    return counts

# 7️⃣ Candidate keywords
all_keywords_candidate = [
    'great', 'amazing', 'excellent', 'wonderful', 'best', 'love', 'fun', 'perfect',
    'terrible', 'awful', 'worst', 'boring', 'bad', 'poor', 'hate', 'disappointing'
]

# 8️⃣ Count review-level presence
pos_review_counts = keyword_review_counts(positive_reviews, all_keywords_candidate)
neg_review_counts = keyword_review_counts(negative_reviews, all_keywords_candidate)

# 9️⃣ Select top 3 positive & negative keywords safely
top_positive = sorted(pos_review_counts, key=pos_review_counts.get, reverse=True)[:3]
top_negative = sorted(neg_review_counts, key=neg_review_counts.get, reverse=True)[:3]

print("Top Positive Keywords:", top_positive)
print("Top Negative Keywords:", top_negative)

# 🔟 Show counts for top keywords
all_top_keywords = top_positive + top_negative
pos_review_counts_top = {k: pos_review_counts[k] for k in all_top_keywords}
neg_review_counts_top = {k: neg_review_counts[k] for k in all_top_keywords}

print("\nNumber of Positive Reviews containing each keyword:", pos_review_counts_top)
print("Number of Negative Reviews containing each keyword:", neg_review_counts_top)

# Bayes Probability Computation

# Use the specific order of keywords
keywords = ['great', 'best', 'love', 'bad', 'worst']

# Prior probability
P_positive = len(positive_reviews) / len(df)

# Prepare list to store results
results = []

# Loop through each keyword
for k in keywords:
    # Number of reviews containing the keyword
    pos_count = pos_review_counts[k]
    neg_count = neg_review_counts[k]
    total_count = pos_count + neg_count

    # Likelihood: P(keyword | Positive)
    likelihood = pos_count / len(positive_reviews)

    # Marginal: P(keyword)
    marginal = total_count / len(df)

    # Posterior: P(Positive | keyword) using Bayes theorem
    posterior = (likelihood * P_positive) / marginal if marginal != 0 else 0

    # Store results with rounded values for better readability
    results.append({
        'Keyword': k,
        'Prior P(Positive)': round(P_positive, 4),
        'Likelihood P(keyword|Positive)': round(likelihood, 4),
        'Marginal P(keyword)': round(marginal, 4),
        'Posterior P(Positive|keyword)': round(posterior, 4)
    })

# Convert results to DataFrame
bayes_table = pd.DataFrame(results)

# Display the table
bayes_table

Columns: ['review', 'sentiment']
                                              review sentiment
0  One of the other reviewers has mentioned that ...  positive
1  A wonderful little production. <br /><br />The...  positive
2  I thought this was a wonderful way to spend ti...  positive
3  Basically there's a family where a little boy ...  negative
4  Petter Mattei's "Love in the Time of Money" is...  positive
Top Positive Keywords: ['great', 'best', 'love']
Top Negative Keywords: ['bad', 'great', 'worst']

Number of Positive Reviews containing each keyword: {'great': 8478, 'best': 6239, 'love': 5666, 'bad': 2961, 'worst': 411}
Number of Negative Reviews containing each keyword: {'great': 4099, 'best': 3430, 'love': 3258, 'bad': 8827, 'worst': 4023}


,Keyword,Prior P(Positive),Likelihood P(keyword|Positive),Marginal P(keyword),Posterior P(Positive|keyword)
0,great,0.5,0.3391,0.2515,0.6741
1,best,0.5,0.2496,0.1934,0.6453
2,love,0.5,0.2266,0.1785,0.6349
3,bad,0.5,0.1184,0.2358,0.2512
4,worst,0.5,0.0164,0.0887,0.0927
